# RAGAS Evaluation Notebook - Single-Turn and Multi-Turn

This merged notebook evaluates the three Excel inputs only:

1. Easy Travel single-turn sheet
2. PolicyBot single-turn sheet
3. PolicyBot multi-turn sheet

The flow is: load shared setup, configure Azure OpenAI once, upload the three sheets once, run single-turn predefined metrics, run single-turn custom metrics, run multi-turn predefined metrics, then run multi-turn custom metrics.

## 1. Install dependencies

Run this once per session. If the packages are already installed, you can skip this cell.

In [ ]:
%pip install \
    ragas==0.3.9 \
    langchain==0.3.27 \
    langchain-openai==0.3.28 \
    langchain-community==0.3.27 \
    langchain-huggingface==0.1.2 \
    openai==1.99.5 \
    datasets==4.0.0 \
    pandas==2.2.2 \
    sentence-transformers==3.4.1 \
    nest_asyncio==1.6.0 \
    openpyxl==3.1.5
print("Install complete. If running on Colab, restart the runtime before continuing.")

## 2. Imports and shared helpers

These helpers load Excel workbooks, normalize common column names, and build RAGAS datasets from the three uploaded sheets. They do not contain sample data.

In [ ]:
import ast
import os
from pathlib import Path

import nest_asyncio
import pandas as pd
from IPython.display import display

from ragas import SingleTurnSample, EvaluationDataset, evaluate
from ragas.dataset_schema import MultiTurnSample
from ragas.messages import HumanMessage, AIMessage

nest_asyncio.apply()

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

SINGLE_TURN_COLUMN_ALIASES = {
    "user_input": ["question", "user_input", "input", "query", "prompt"],
    "reference": ["ground_truth", "groundtruth", "reference", "expected_output", "expected", "gold_answer", "expected_answer"],
    "response": ["answer", "response", "actual_output", "output", "prediction", "model_answer", "actual_answer", "generated_output"],
    "retrieved_contexts": ["contexts", "context", "retrieved_contexts", "retrieval_context", "retrieved_context", "actual_context", "actual_contex", "actual_contexts"],
}
CONTEXT_DELIMITERS = ["||", "\n", ";"]

MULTI_TURN_COLUMN_ALIASES = {
    "input": "question",
    "user_input": "question",
    "prompt": "question",
    "actual_output": "actual_answer",
    "generated_output": "actual_answer",
    "output": "actual_answer",
    "response": "actual_answer",
    "answer": "actual_answer",
    "expected_output": "expected_answer",
    "expected": "expected_answer",
    "reference": "expected_answer",
    "ground_truth": "expected_answer",
}


def _norm(name):
    return str(name).strip().lower().replace(" ", "_")


def _is_blank(value):
    return value is None or (isinstance(value, float) and pd.isna(value)) or str(value).strip() == ""


def _text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def upload_excel(label):
    print(f"Upload the {label} Excel file (.xlsx)")
    if IN_COLAB:
        uploaded = files.upload()
        excel_path = next(iter(uploaded))
    else:
        excel_path = input(f"Path to the {label} Excel file: ").strip().strip('"')
    if not excel_path:
        raise ValueError(f"No Excel file was provided for {label}.")
    df = pd.read_excel(excel_path)
    print(f'Loaded "{excel_path}" with {len(df)} row(s).')
    print("Columns found:", list(df.columns))
    return excel_path, df


def build_single_turn_field_map(columns):
    norm_cols = {_norm(c): c for c in columns}
    field_map = {}
    for field, aliases in SINGLE_TURN_COLUMN_ALIASES.items():
        for alias in aliases:
            if alias in norm_cols:
                field_map[field] = norm_cols[alias]
                break
    return field_map


def split_contexts(value):
    if _is_blank(value):
        return None
    text = str(value).strip()
    for delim in CONTEXT_DELIMITERS:
        if delim in text:
            parts = [part.strip() for part in text.split(delim) if part.strip()]
            return parts or None
    return [text]


def excel_to_single_turn_dataset(df, label):
    field_map = build_single_turn_field_map(df.columns)
    print(f"\n{label}: detected column -> RAGAS field")
    for field, col in field_map.items():
        print(f"  {col!r} -> {field}")

    required = {"user_input", "response"}
    missing_required = required - set(field_map)
    if missing_required:
        raise ValueError(f"{label} is missing required single-turn field(s): {sorted(missing_required)}")

    samples = []
    for _, row in df.iterrows():
        kwargs = {}
        for field, col in field_map.items():
            value = row.get(col)
            if field == "retrieved_contexts":
                contexts = split_contexts(value)
                if contexts is not None:
                    kwargs[field] = contexts
            elif not _is_blank(value):
                kwargs[field] = str(value).strip()
        if kwargs:
            samples.append(SingleTurnSample(**kwargs))

    if not samples:
        raise ValueError(f"No usable single-turn samples were found in {label}.")

    print(f"Built {len(samples)} single-turn sample(s) from {len(df)} row(s).")
    return EvaluationDataset(samples=samples), set(field_map)


def normalize_multi_turn_columns(df):
    normalized = df.copy()
    normalized.columns = [_norm(col) for col in normalized.columns]
    rename_map = {
        source: target
        for source, target in MULTI_TURN_COLUMN_ALIASES.items()
        if source in normalized.columns and target not in normalized.columns
    }
    return normalized.rename(columns=rename_map)


def parse_ragas_messages(value):
    text = _text(value)
    if not text:
        return []

    tree = ast.parse(text, mode="eval")
    if not isinstance(tree.body, (ast.List, ast.Tuple)):
        raise ValueError("The ragas cell must be a list of HumanMessage(...) and AIMessage(...).")

    messages = []
    for node in tree.body.elts:
        if not isinstance(node, ast.Call) or not isinstance(node.func, ast.Name):
            raise ValueError("Each ragas entry must be HumanMessage(...) or AIMessage(...).")
        if node.func.id not in {"HumanMessage", "AIMessage"}:
            raise ValueError(f"Unsupported ragas message type: {node.func.id}")

        kwargs = {kw.arg: kw.value for kw in node.keywords if kw.arg}
        content_node = kwargs.get("content")
        if not isinstance(content_node, ast.Constant) or not isinstance(content_node.value, str):
            raise ValueError("Each ragas message must have a string content=... value.")

        message_cls = HumanMessage if node.func.id == "HumanMessage" else AIMessage
        messages.append(message_cls(content=content_node.value))

    return messages


def build_reference(group):
    if "expected_answer" not in group.columns:
        return ""
    parts = []
    for _, row in group.iterrows():
        expected = _text(row.get("expected_answer"))
        question = _text(row.get("question")) if "question" in group.columns else ""
        if expected:
            prefix = f"Q: {question}\n" if question else ""
            parts.append(f"{prefix}Expected answer: {expected}")
    return "\n\n".join(parts)


def build_messages_from_rows(group):
    messages = []
    for _, row in group.iterrows():
        question = _text(row.get("question")) if "question" in group.columns else ""
        actual = _text(row.get("actual_answer")) if "actual_answer" in group.columns else ""
        if question:
            messages.append(HumanMessage(content=question))
        if actual:
            messages.append(AIMessage(content=actual))
    return messages


def excel_to_multi_turn_dataset(df, label):
    raw_df = normalize_multi_turn_columns(df)
    if not ({"ragas"} & set(raw_df.columns) or {"question", "actual_answer"}.issubset(raw_df.columns)):
        raise ValueError(
            f"{label} must contain a 'ragas' column, or at least 'question' and 'actual_answer' columns."
        )

    if "case_id" not in raw_df.columns:
        raw_df["case_id"] = [f"ROW-{idx:04d}" for idx in range(len(raw_df))]
    if "turn" in raw_df.columns:
        raw_df["turn_sort"] = pd.to_numeric(raw_df["turn"], errors="coerce")
    else:
        raw_df["turn_sort"] = raw_df.groupby("case_id").cumcount() + 1

    raw_df = raw_df.sort_values(["case_id", "turn_sort"], kind="stable")

    samples = []
    sample_rows = []
    for case_id, group in raw_df.groupby("case_id", sort=False):
        ragas_messages = []
        if "ragas" in group.columns:
            for _, row in group.iterrows():
                if _text(row.get("ragas")):
                    ragas_messages = parse_ragas_messages(row.get("ragas"))
                    break

        messages = ragas_messages or build_messages_from_rows(group)
        if not messages:
            continue

        reference = build_reference(group)
        sample_kwargs = {"user_input": messages}
        if reference:
            sample_kwargs["reference"] = reference

        sample = MultiTurnSample(**sample_kwargs)
        samples.append(sample)


        sample_rows.append(
            {
                "case_id": case_id,
                "turns": len(messages),
                "source": "ragas" if ragas_messages else "question_actual_answer",
                "has_reference": bool(reference),
            }
        )

    if not samples:
        raise ValueError(f"No usable multi-turn conversations were found in {label}.")

    summary = pd.DataFrame(sample_rows)
    print(f"Built {len(samples)} multi-turn conversation sample(s).")
    display(summary)
    return EvaluationDataset(samples=samples), summary

print("Imports and helpers ready.")

## 3. Configure Azure OpenAI credentials

This is the only credential/configuration cell in the notebook. Fill in all Azure OpenAI values here once; every metric section reuses these objects.

In [ ]:
import os

AZURE_OPENAI_API_KEY = "YOUR_AZURE_OPENAI_API_KEY"
AZURE_OPENAI_API_VERSION = "YOUR_AZURE_OPENAI_API_VERSION"
AZURE_OPENAI_ENDPOINT = "YOUR_AZURE_OPENAI_ENDPOINT"
AZURE_OPENAI_DEPLOYMENT_NAME = "YOUR_AZURE_OPENAI_DEPLOYMENT_NAME"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

os.environ["AZURE_OPENAI_API_KEY"] = AZURE_OPENAI_API_KEY
os.environ["AZURE_OPENAI_API_VERSION"] = AZURE_OPENAI_API_VERSION
os.environ["AZURE_OPENAI_ENDPOINT"] = AZURE_OPENAI_ENDPOINT
os.environ["AZURE_DEPLOYMENT_NAME"] = AZURE_OPENAI_DEPLOYMENT_NAME

from langchain_openai import AzureChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms.base import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

class GPT5LangchainLLMWrapper(LangchainLLMWrapper):
    def generate_text(self, prompt, n=1, temperature=None, stop=None, callbacks=None):
        return super().generate_text(prompt, n=n, temperature=1, stop=stop, callbacks=callbacks)

    async def agenerate_text(self, prompt, n=1, temperature=None, stop=None, callbacks=None):
        return await super().agenerate_text(prompt, n=n, temperature=1, stop=stop, callbacks=callbacks)

llm_client = AzureChatOpenAI(
    azure_deployment=AZURE_OPENAI_DEPLOYMENT_NAME,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=1,
)
wrapped_llm = GPT5LangchainLLMWrapper(llm_client, bypass_temperature=True)
evaluator_llm = wrapped_llm

evaluator_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
)

print("API key present?", bool(AZURE_OPENAI_API_KEY and AZURE_OPENAI_API_KEY != "YOUR_AZURE_OPENAI_API_KEY"))
print("Endpoint:", AZURE_OPENAI_ENDPOINT)
print("Deployment:", AZURE_OPENAI_DEPLOYMENT_NAME)
print("RAGAS LLM and embeddings ready.")

## 4. Upload Easy Travel single-turn sheet

In [ ]:
easytravel_single_turn_path, easytravel_single_turn_df = upload_excel("Easy Travel single-turn")
easytravel_single_turn_dataset, easytravel_single_turn_fields = excel_to_single_turn_dataset(
    easytravel_single_turn_df,
    "Easy Travel single-turn",
)

## 5. Upload PolicyBot single-turn sheet

In [ ]:
policybot_single_turn_path, policybot_single_turn_df = upload_excel("PolicyBot single-turn")
policybot_single_turn_dataset, policybot_single_turn_fields = excel_to_single_turn_dataset(
    policybot_single_turn_df,
    "PolicyBot single-turn",
)

## 6. Upload PolicyBot multi-turn sheet

In [ ]:
policybot_multi_turn_path, policybot_multi_turn_df = upload_excel("PolicyBot multi-turn")
policybot_multi_turn_dataset, policybot_multi_turn_summary = excel_to_multi_turn_dataset(
    policybot_multi_turn_df,
    "PolicyBot multi-turn",
)

# Single-Turn Evaluation

## 7. Single-turn predefined metrics

Easy Travel uses no-context metrics. PolicyBot uses the full RAG metric set when the sheet includes retrieved contexts.

In [ ]:
from ragas.metrics import (
    AnswerCorrectness,
    AnswerSimilarity,
    Faithfulness,
    ContextRecall,
    ContextPrecision,
    NoiseSensitivity,
)

answer_similarity = AnswerSimilarity(embeddings=evaluator_embeddings)

single_turn_no_context_metrics = [
    AnswerCorrectness(llm=wrapped_llm, answer_similarity=answer_similarity),
    answer_similarity,
]

single_turn_rag_metrics = [
    AnswerCorrectness(llm=wrapped_llm, answer_similarity=answer_similarity),
    answer_similarity,
    Faithfulness(llm=wrapped_llm),
    ContextRecall(llm=wrapped_llm),
    ContextPrecision(llm=wrapped_llm),
    NoiseSensitivity(llm=wrapped_llm),
]

policybot_single_turn_predefined_metrics = (
    single_turn_rag_metrics
    if "retrieved_contexts" in policybot_single_turn_fields
    else single_turn_no_context_metrics
)

print("Easy Travel predefined metrics:", [getattr(metric, "name", metric.__class__.__name__) for metric in single_turn_no_context_metrics])
print("PolicyBot predefined metrics:", [getattr(metric, "name", metric.__class__.__name__) for metric in policybot_single_turn_predefined_metrics])

In [ ]:
easytravel_single_turn_predefined_result = evaluate(
    dataset=easytravel_single_turn_dataset,
    metrics=single_turn_no_context_metrics,
    llm=wrapped_llm,
    embeddings=evaluator_embeddings,
)
easytravel_single_turn_predefined_df = easytravel_single_turn_predefined_result.to_pandas()
display(easytravel_single_turn_predefined_df)
easytravel_single_turn_predefined_df.to_excel("easytravel_single_turn_predefined_ragas_results.xlsx", index=False)
print("Saved easytravel_single_turn_predefined_ragas_results.xlsx")

In [ ]:
policybot_single_turn_predefined_result = evaluate(
    dataset=policybot_single_turn_dataset,
    metrics=policybot_single_turn_predefined_metrics,
    llm=wrapped_llm,
    embeddings=evaluator_embeddings,
)
policybot_single_turn_predefined_df = policybot_single_turn_predefined_result.to_pandas()
display(policybot_single_turn_predefined_df)
policybot_single_turn_predefined_df.to_excel("policybot_single_turn_predefined_ragas_results.xlsx", index=False)
print("Saved policybot_single_turn_predefined_ragas_results.xlsx")

## 8. Single-turn custom metrics

These AspectCritic metrics are applied directly to the uploaded single-turn sheets.

In [ ]:
from ragas.metrics import AspectCritic

single_turn_answer_correctness_critic = AspectCritic(
    name="answer_correctness_binary",
    definition="Return 1 if the answer correctly conveys the same meaning as the reference answer; otherwise return 0.",
    llm=wrapped_llm,
)

single_turn_fluency = AspectCritic(
    name="fluency_binary",
    definition="Return 1 if the answer is grammatically correct and easy to understand; otherwise return 0.",
    llm=wrapped_llm,
)

single_turn_completeness = AspectCritic(
    name="completeness_binary",
    definition="Return 1 if the answer fully addresses the question using the available reference or context; otherwise return 0.",
    llm=wrapped_llm,
)

single_turn_conciseness = AspectCritic(
    name="conciseness_binary",
    definition="Return 1 if the answer is direct and avoids unnecessary or repeated information; otherwise return 0.",
    llm=wrapped_llm,
)

single_turn_groundedness = AspectCritic(
    name="groundedness_binary",
    definition="Return 1 if every answer claim is supported by the provided context or reference; otherwise return 0.",
    llm=wrapped_llm,
)

single_turn_custom_metrics = [
    single_turn_answer_correctness_critic,
    single_turn_fluency,
    single_turn_completeness,
    single_turn_conciseness,
    single_turn_groundedness,
]
print("Single-turn custom metrics:", [metric.name for metric in single_turn_custom_metrics])

In [ ]:
easytravel_single_turn_custom_result = evaluate(
    dataset=easytravel_single_turn_dataset,
    metrics=single_turn_custom_metrics,
    llm=wrapped_llm,
)
easytravel_single_turn_custom_df = easytravel_single_turn_custom_result.to_pandas()
display(easytravel_single_turn_custom_df)
easytravel_single_turn_custom_df.to_excel("easytravel_single_turn_custom_ragas_results.xlsx", index=False)
print("Saved easytravel_single_turn_custom_ragas_results.xlsx")

In [ ]:
policybot_single_turn_custom_result = evaluate(
    dataset=policybot_single_turn_dataset,
    metrics=single_turn_custom_metrics,
    llm=wrapped_llm,
)
policybot_single_turn_custom_df = policybot_single_turn_custom_result.to_pandas()
display(policybot_single_turn_custom_df)
policybot_single_turn_custom_df.to_excel("policybot_single_turn_custom_ragas_results.xlsx", index=False)
print("Saved policybot_single_turn_custom_ragas_results.xlsx")

# Multi-Turn Evaluation

## 9. Multi-turn predefined metrics

These RAGAS predefined metrics run against the uploaded PolicyBot multi-turn sheet. `AgentGoalAccuracyWithReference` is included only when every conversation has an expected answer/reference.

In [ ]:
from ragas.metrics import (
    AgentGoalAccuracyWithoutReference,
    AgentGoalAccuracyWithReference,
)

multi_turn_predefined_metrics = [
    AgentGoalAccuracyWithoutReference(llm=evaluator_llm),
]

if policybot_multi_turn_summary["has_reference"].all():
    multi_turn_predefined_metrics.append(AgentGoalAccuracyWithReference(llm=evaluator_llm))
else:
    print("Skipping AgentGoalAccuracyWithReference because at least one conversation has no expected answer/reference.")

print("PolicyBot multi-turn predefined metrics:", [getattr(metric, "name", metric.__class__.__name__) for metric in multi_turn_predefined_metrics])

In [ ]:
policybot_multi_turn_predefined_result = evaluate(
    dataset=policybot_multi_turn_dataset,
    metrics=multi_turn_predefined_metrics,
)
policybot_multi_turn_predefined_scores_df = policybot_multi_turn_predefined_result.to_pandas()
policybot_multi_turn_predefined_df = pd.concat(
    [policybot_multi_turn_summary.reset_index(drop=True), policybot_multi_turn_predefined_scores_df.reset_index(drop=True)],
    axis=1,
)
display(policybot_multi_turn_predefined_df)
policybot_multi_turn_predefined_df.to_excel("policybot_multi_turn_predefined_ragas_results.xlsx", index=False)
print("Saved policybot_multi_turn_predefined_ragas_results.xlsx")

## 10. Multi-turn SimpleCriteriaScore metrics

These two graded custom rubric metrics score from 0 to 1 and are applied directly to the uploaded PolicyBot multi-turn sheet.

In [ ]:
from ragas.metrics import SimpleCriteriaScore

multi_turn_groundedness = SimpleCriteriaScore(
    name="multi_turn_groundedness_0_to_1",
    definition=(
        "Score from 0 to 1 how well assistant responses are grounded in the conversation and avoid unsupported facts. "
        "Use partial credit: 0 = mostly unsupported, 0.5 = partially grounded, 1 = fully grounded."
    ),
    llm=evaluator_llm,
)

multi_turn_answer_quality = SimpleCriteriaScore(
    name="multi_turn_answer_quality_0_to_1",
    definition=(
        "Score from 0 to 1 how factually correct, complete, context-consistent, and useful the assistant responses are. "
        "Use partial credit for answers that are directionally correct but incomplete or missing details."
    ),
    llm=evaluator_llm,
)

multi_turn_custom_metrics = [
    multi_turn_groundedness,
    multi_turn_answer_quality,
]
print("PolicyBot multi-turn SimpleCriteriaScore metrics:", [metric.name for metric in multi_turn_custom_metrics])

In [ ]:
policybot_multi_turn_custom_result = evaluate(
    dataset=policybot_multi_turn_dataset,
    metrics=multi_turn_custom_metrics,
)
policybot_multi_turn_custom_scores_df = policybot_multi_turn_custom_result.to_pandas()
policybot_multi_turn_custom_df = pd.concat(
    [policybot_multi_turn_summary.reset_index(drop=True), policybot_multi_turn_custom_scores_df.reset_index(drop=True)],
    axis=1,
)
display(policybot_multi_turn_custom_df)
policybot_multi_turn_custom_df.to_excel("policybot_multi_turn_custom_ragas_results.xlsx", index=False)
print("Saved policybot_multi_turn_custom_ragas_results.xlsx")

## 11. Output files

The notebook writes separate Excel reports for each section so you can review predefined and custom metric results independently.